In [1]:
#import numpy as np
#import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn import mixture
import scanpy as sc
import pandas as pd
#import pomegranate

In [2]:
!nvidia-smi

Tue Mar 17 12:07:15 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 520.56.06    Driver Version: 525.125.06   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A6000    Off  | 00000000:01:00.0 Off |                  Off |
| 32%   61C    P2    91W / 300W |   2580MiB / 49140MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A6000    Off  | 00000000:24:00.0 Off |                  Off |
| 30%   

In [3]:
import os
import sys
import numpy as np
#sys.path.append('/banach2/sennet/locat/')
sys.path.append('../../../')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

SEED = 13

def reset_seeds(seed=SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    return seed

reset_seeds(SEED)


13

In [4]:
import numpy as np
import os

# Point rpy2 at the R in your env (adjust if your env path differs)
os.environ["R_HOME"] = "/banach2/wes/envs/lmd_rpy2/lib/R"

# Ensure base packages are attached on startup
os.environ["R_DEFAULT_PACKAGES"] = "base,utils,stats,graphics,grDevices,methods"

# (Optional) keep your user lib too — just make sure it doesn’t hide base
# os.environ["R_LIBS_USER"] = os.path.expanduser("~/R/4.3/library")

# Now import rpy2-dependent code

In [5]:
from rpy2.robjects import conversion, default_converter
from rpy2.robjects import numpy2ri

# Set global conversion (recommended for notebooks using rpy2 + numpy)
conversion.set_conversion(default_converter + numpy2ri.converter)

# Now import your helper
from locat.run_lmd import lmd_scores_from_adata


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

In [6]:
import numpy as np

In [7]:
from tqdm import tqdm

from matplotlib import pyplot as plt
import seaborn as sns
from collections import Counter

In [8]:
import h5py
import numpy as np
data_dir = ""
fn = data_dir+"support_files/method_comp_1.h5ad"

with h5py.File(fn, "r+") as f:
    # Confirm the path exists
    if "/uns/log1p/base" in f:
        # Delete and recreate as a scalar float64 dataset
        del f["/uns/log1p/base"]
        f.create_dataset("/uns/log1p/base", data=np.float64(2.0))
        print("Patched /uns/log1p/base to 2.0")
    else:
        print("No /uns/log1p/base found; nothing to patch.")

No /uns/log1p/base found; nothing to patch.


In [9]:
%%time
import scanpy as sc
data_dir = ""
adata = sc.read_h5ad(data_dir+'support_files/method_comp_1.h5ad')

CPU times: user 22.6 ms, sys: 19.9 ms, total: 42.5 ms
Wall time: 948 ms


In [ ]:
#plt.hist(np.array(adata.X.todense()).flatten()[np.array(adata.X.todense()).flatten()>0], bins=30)

In [ ]:
X = adata.X#.todense()

# Summary statistics
print(f"Type: {type(X)}")
print(f"Dtype: {X.dtype}")
print(f"Min: {X.min()}, Max: {X.max()}")
#print(f"Mean: {X.mean()}, Median: {np.median(X.A if hasattr(X, 'A') else X)}")

In [ ]:
#sc.pp.log1p(adata)

In [ ]:
#plt.hist(np.array(adata.X.todense()).flatten()[np.array(adata.X.todense()).flatten()>0], bins=30)

In [ ]:
adata

In [ ]:
# get top m HVG
#sc.pp.highly_variable_genes(adata, n_top_genes=4000, flavor='cell_ranger', batch_key = "pool_id")

In [ ]:
#hvg_df = adata.var[adata.var['highly_variable']]

In [ ]:
adata_train = adata#[:,np.array(hvg_df.index)].copy()

In [ ]:
#adata_train.X = adata_train.X.toarray()

In [ ]:
adata_train

In [ ]:
adata.obs_names = adata.obs_names.astype(str)
adata.var_names = adata.var_names.astype(str)

In [ ]:
adata.obsm["pca"] = adata.obsm["coords"]

In [ ]:
%%time
from locat.run_lmd import prep_W_from_adata_connectivities, lmd_scores_from_adata

# 1) Prep the graph from Scanpy neighbors
W_csr = prep_W_from_adata_connectivities(
    adata_train,
    k=5,           # lighter than 5; good for 150k cells
    binarize=False # keep weights; set True if you prefer unweighted kNN
)

# 2) Run coarse LMD using your PCA + precomputed graph
#scores = lmd_scores_coarse_from_adata_preknn(
#    adata_train,
#    feature_space_key="pca",   # or "X_pca"
#    W_csr=W_csr,               # << uses your existing graph
#    min_cells=5,
#    coarse_N=5000,             # start safer; can try 10k later
#    max_time=2**10,            # 1024; bump if you want more diffusion
#    assume_counts=False,
#    center_scale_pcs=False,
#    debug=True
#)

reset_seeds(SEED)
scores = lmd_scores_from_adata(
    adata_train,
    feature_space_key="pca",   # or "X_pca"
    #W_csr=W_csr,               # << uses your existing graph
    min_cells=5,
    #coarse_N=5000,             # start safer; can try 10k later
    max_time=2**10,            # 1024; bump if you want more diffusion
    assume_counts=False,
    center_scale_pcs=False,
    debug=True
)


adata_train.var["LMD_score"] = scores.reindex(adata_train.var_names).to_numpy()
scores.sort_values(ascending=False).head(10)


In [ ]:
adata_train

In [ ]:
#pip install cloudpickle

In [ ]:
import cloudpickle as cp
def save_res(path):
    with open(f"{path}", "wb") as f:
        cp.dump(adata_train.var["LMD_score"], f)

def load_res(path):
    with open(f"{path}", "rb") as f:
        loadedres = cp.load(f)
    return loadedres

In [ ]:
save_res(path = data_dir+'LMD_output_comp1_repro.pkl')

In [28]:
np.savez(data_dir+'LMD_output_comp1_repro.npz', var_names=np.array(adata_train.var_names), gene_localization=adata_train.var["LMD_score"])

In [29]:
data = np.load(data_dir+'LMD_output_comp1_repro.npz', allow_pickle=True)
#var_names = data["var_names"]
#gene_localization = data["gene_localization"]

In [30]:
lmd_score2 = load_res(path = data_dir+'LMD_output_comp1_repro.pkl')

In [31]:
lmd_score2

Gene_0      5.731932
Gene_1      5.806738
Gene_2      5.827487
Gene_3      5.680928
Gene_4      5.870095
              ...   
Gene_195    7.203338
Gene_196    7.249066
Gene_197    7.368406
Gene_198    7.230244
Gene_199    7.324248
Name: LMD_score, Length: 200, dtype: float64

# Next Simulation

In [32]:
import scanpy as sc
data_dir = ""
adata = sc.read_h5ad(data_dir+"support_files/method_comp_2.h5ad") 

In [33]:
adata.obs_names = adata.obs_names.astype(str)
adata.var_names = adata.var_names.astype(str)

In [34]:
adata.obsm["pca"] = adata.obsm["coords"]

In [35]:
%%time
from locat.run_lmd import prep_W_from_adata_connectivities, lmd_scores_from_adata

# 1) Prep the graph from Scanpy neighbors
W_csr = prep_W_from_adata_connectivities(
    adata,
    k=5,           # lighter than 5; good for 150k cells
    binarize=False # keep weights; set True if you prefer unweighted kNN
)

reset_seeds(SEED)
scores = lmd_scores_from_adata(
    adata,
    feature_space_key="pca",   # or "X_pca"
    #W_csr=W_csr,               # << uses your existing graph
    min_cells=5,
    #coarse_N=5000,             # start safer; can try 10k later
    max_time=2**10,            # 1024; bump if you want more diffusion
    assume_counts=False,
    center_scale_pcs=False,
    debug=True
)


adata.var["LMD_score"] = scores.reindex(adata.var_names).to_numpy()
scores.sort_values(ascending=False).head(10)


R callback write-console: Final dims: dat (genes x cells) = 200 x 5000; feature_space (cells x dims) = 5000 x 2
  
R callback write-console: Names aligned (cells): TRUE
  


Pre-validate R checks:
  has rownames(dat)? True
  has colnames(dat)? True
  has rownames(feature_space)? True
Names aligned (cells): True
R dims dat (genes x cells): 200 x 5000, feature_space: 5000 x 2
Constructing KNN graph
Remove 0 genes which express in less than 5 cells
Calculate LMD score profile for large data...
Run doubly stochastic on affinity matrix...

max diffusion time:2^ 10 
CPU times: user 6min 39s, sys: 5.84 s, total: 6min 45s
Wall time: 41.4 s


Gene_177    7.438475
Gene_182    7.436934
Gene_184    7.419612
Gene_172    7.414352
Gene_197    7.368406
Gene_193    7.350814
Gene_186    7.347631
Gene_173    7.328902
Gene_199    7.324248
Gene_183    7.318246
Name: LMD_score, dtype: float64

In [36]:
import cloudpickle as cp
def save_res(path):
    with open(f"{path}", "wb") as f:
        cp.dump(adata.var["LMD_score"], f)

def load_res(path):
    with open(f"{path}", "rb") as f:
        loadedres = cp.load(f)
    return loadedres

In [37]:
save_res(path = data_dir+'LMD_output_comp2_repro.pkl')

In [38]:
np.savez(data_dir+'LMD_output_comp2_repro.npz', var_names=np.array(adata.var_names), gene_localization=adata.var["LMD_score"])

In [39]:
data = np.load(data_dir+'LMD_output_comp2_repro.npz', allow_pickle=True)

In [40]:
lmd_score2 = load_res(path = data_dir+'LMD_output_comp2_repro.pkl')

In [ ]:
lmd_score2

# Next Simulation

In [10]:
%%time
import scanpy as sc
data_dir = ""
adata = sc.read_h5ad(data_dir+"support_files/sim_subs.h5ad")

CPU times: user 59.6 ms, sys: 15.4 ms, total: 75 ms
Wall time: 375 ms


In [11]:
adata.obs_names = adata.obs_names.astype(str)
adata.var_names = adata.var_names.astype(str)

In [12]:
#adata.obsm["pca"] = adata.obsm["coords"]

In [13]:
%%time
from locat.run_lmd import prep_W_from_adata_connectivities, lmd_scores_from_adata

# 1) Prep the graph from Scanpy neighbors
W_csr = prep_W_from_adata_connectivities(
    adata,
    k=5,           # lighter than 5; good for 150k cells
    binarize=False # keep weights; set True if you prefer unweighted kNN
)

reset_seeds(SEED)
scores = lmd_scores_from_adata(
    adata,
    feature_space_key="pca",   # or "X_pca"
    #W_csr=W_csr,               # << uses your existing graph
    min_cells=5,
    #coarse_N=5000,             # start safer; can try 10k later
    max_time=2**10,            # 1024; bump if you want more diffusion
    assume_counts=False,
    center_scale_pcs=False,
    debug=True
)


adata.var["LMD_score"] = scores.reindex(adata.var_names).to_numpy()
scores.sort_values(ascending=False).head(10)


Pre-validate R checks:
  has rownames(dat)? True
  has colnames(dat)? True
  has rownames(feature_space)? True
Names aligned (cells): True
R dims dat (genes x cells): 1100 x 5572, feature_space: 5572 x 50


R callback write-console: Final dims: dat (genes x cells) = 1100 x 5572; feature_space (cells x dims) = 5572 x 50
  
R callback write-console: Names aligned (cells): TRUE
  


Constructing KNN graph
Remove 0 genes which express in less than 5 cells
Calculate LMD score profile for large data...
Run doubly stochastic on affinity matrix...

max diffusion time:2^ 10 
CPU times: user 11min 13s, sys: 10.4 s, total: 11min 23s
Wall time: 1min 6s


Cops9            9.699337
Dguok            9.692838
Mrpl53           9.683513
0610012G03Rik    9.682369
Mrpl21           9.678409
Eif3g            9.676803
Nfu1             9.665043
Ccdc59           9.664123
2810004N23Rik    9.663282
Ubxn1            9.659400
Name: LMD_score, dtype: float64

In [14]:
import cloudpickle as cp
def save_res(path):
    with open(f"{path}", "wb") as f:
        cp.dump(adata.var["LMD_score"], f)

def load_res(path):
    with open(f"{path}", "rb") as f:
        loadedres = cp.load(f)
    return loadedres

In [15]:
save_res(path = data_dir+'support_files/LMD_output_subs_comp_repro.pkl')

In [16]:
np.savez(data_dir+'support_files/LMD_output_subs_comp_repro.npz', var_names=np.array(adata.var_names), gene_localization=adata.var["LMD_score"])

In [17]:
data = np.load(data_dir+'support_files/LMD_output_subs_comp_repro.npz', allow_pickle=True)

In [18]:
lmd_score2 = load_res(path = data_dir+'support_files/LMD_output_subs_comp_repro.pkl')

In [19]:
lmd_score2

Timm8a1         9.292311
Otud7a          8.624225
Pde4dip         9.418986
Ccdc178         8.268684
Rab11a          9.623533
                  ...   
Sox2_sub_095    6.317907
Sox2_sub_096    6.406302
Sox2_sub_097    6.551538
Sox2_sub_098    6.692028
Sox2_sub_099    6.541014
Name: LMD_score, Length: 1100, dtype: float64

# Next Simulation

In [20]:
%%time
import scanpy as sc
data_dir = ""
adata = sc.read_h5ad(data_dir+"support_files/sim_depletion.h5ad")

CPU times: user 84.9 ms, sys: 15.3 ms, total: 100 ms
Wall time: 4.94 s


In [21]:
adata.obs_names = adata.obs_names.astype(str)
adata.var_names = adata.var_names.astype(str)

In [22]:
%%time
from locat.run_lmd import prep_W_from_adata_connectivities, lmd_scores_from_adata

# 1) Prep the graph from Scanpy neighbors
W_csr = prep_W_from_adata_connectivities(
    adata,
    k=5,           # lighter than 5; good for 150k cells
    binarize=False # keep weights; set True if you prefer unweighted kNN
)

reset_seeds(SEED)
scores = lmd_scores_from_adata(
    adata,
    feature_space_key="pca",   # or "X_pca"
    #W_csr=W_csr,               # << uses your existing graph
    min_cells=5,
    #coarse_N=5000,             # start safer; can try 10k later
    max_time=2**10,            # 1024; bump if you want more diffusion
    assume_counts=False,
    center_scale_pcs=False,
    debug=True
)


adata.var["LMD_score"] = scores.reindex(adata.var_names).to_numpy()
scores.sort_values(ascending=False).head(10)


Pre-validate R checks:
  has rownames(dat)? True
  has colnames(dat)? True
  has rownames(feature_space)? True
Names aligned (cells): True
R dims dat (genes x cells): 1050 x 5572, feature_space: 5572 x 50


R callback write-console: Final dims: dat (genes x cells) = 1050 x 5572; feature_space (cells x dims) = 5572 x 50
  
R callback write-console: Names aligned (cells): TRUE
  


Constructing KNN graph
Remove 0 genes which express in less than 5 cells
Calculate LMD score profile for large data...
Run doubly stochastic on affinity matrix...

max diffusion time:2^ 10 
CPU times: user 10min 51s, sys: 10 s, total: 11min 1s
Wall time: 1min 2s


Cops9            9.699337
Dguok            9.692838
Mrpl53           9.683513
0610012G03Rik    9.682369
Mrpl21           9.678409
Eif3g            9.676803
Nfu1             9.665043
Ccdc59           9.664123
2810004N23Rik    9.663282
Ubxn1            9.659400
Name: LMD_score, dtype: float64

In [23]:
import cloudpickle as cp
def save_res(path):
    with open(f"{path}", "wb") as f:
        cp.dump(adata.var["LMD_score"], f)

def load_res(path):
    with open(f"{path}", "rb") as f:
        loadedres = cp.load(f)
    return loadedres

In [24]:
save_res(path = data_dir+'support_files/LMD_output_depl_comp_repro.pkl')

In [25]:
np.savez(data_dir+'support_files/LMD_output_depl_comp_repro.npz', var_names=np.array(adata.var_names), gene_localization=adata.var["LMD_score"])

In [26]:
data = np.load(data_dir+'support_files/LMD_output_depl_comp_repro.npz', allow_pickle=True)

In [27]:
lmd_score2 = load_res(path = data_dir+'support_files/LMD_output_depl_comp_repro.pkl')

In [28]:
lmd_score2

Timm8a1           9.292311
Otud7a            8.624225
Pde4dip           9.418986
Ccdc178           8.268684
Rab11a            9.623533
                    ...   
Sox2_bleed_045    8.991600
Sox2_bleed_046    9.012700
Sox2_bleed_047    8.990271
Sox2_bleed_048    9.091808
Sox2_bleed_049    9.047820
Name: LMD_score, Length: 1050, dtype: float64